# Exploring Neural Population Dynamics in the Visual Cortex During Supervised and Unsupervised Learning 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns

# Load Data

In [2]:
# Use a project-rooted path to the data directory
from pathlib import Path
import os

# Resolve project root (works in notebooks): start from CWD and search upward for a folder containing 'data'
base = Path.cwd()
for p in [base, *base.parents]:
    if (p / 'data').exists():
        project_root = p
        break
else:
    project_root = base

root = str(project_root / 'data')

# print("Looking for data in:", root)
# print("Files in data directory:", os.listdir(root))
sup_bef = 'VR2_2021_03_20_1' #example mouse before supervised learning
sup_aft = 'VR2_2021_04_06_1' #example mouse after supervised learning
unsup_bef = 'TX105_2022_10_08_2' #example mouse before supervised learning
unsup_aft = 'TX105_2022_10_19_2' #example mouse after supervised learning

# Load specific files from the directory
spike_file = os.path.join(root, sup_bef + '_SVD_dec.npy')
# print(f"Spike file path: {spike_file}")
beh_path = os.path.join(root, 'Beh_sup_train1_before_learning.npy')
# Load data
svd_ = np.load(spike_file, allow_pickle=True).item()
spikes = svd_['U'].T @ svd_['V']  # project from the PC space back to neural space

# Load and process sound cue data
beh = np.load(beh_path, allow_pickle=True).item()[sup_bef]
beh.keys()

dict_keys(['ntrials', 'trInd', 'trInd_odd', 'trInd_even', 'Trial_start_time', 'Trial_end_time', 'SubjMove', 'Gray_space_time', 'SoundPos', 'SoundTime', 'SoundTimeDelay', 'RewTime', 'RewPos', 'isRew', 'WallType', 'WallIsProbe', 'WallName', 'UniqWalls', 'LickTrind', 'LickTime', 'LickPos', 'Lick_wallName', 'VRposTime', 'VRpos', 'VRposCum', 'ft', 'ft_trInd', 'ft_trInd_odd', 'ft_trInd_even', 'ft_PosCum', 'ft_Pos', 'ft_move', 'ft_isMoving', 'ft_GraySpc', 'ft_CorrSpc', 'ft_WallID', 'ft_RunCum', 'ft_RunSpeed', 'Corridor_Length', 'Gray_Space_length', 'Texture_Length', 'run_pos', 'RewardFr', 'StartFr', 'GrayFr', 'EndFr', 'LickFr', 'SoundFr', 'SoundDelayFr', 'SoundDelPos', 'RunFr', 'BefCueFr', 'AftCueFr', 'stim_id', 'TrialStim', 'StimTrial', 'StimFrame', 'Reward_Mode', 'Reward_Delay_ms'])

# 1. Extract beh information 
We collect arrays with similar length and put them into a DataFrame (df). At the end we have several df with different lengths 

In [3]:
# Collect only those items that are array-like, not strings, and 1-dimensional
array_keys_1d = [k for k, v in beh.items() 
                 if hasattr(v, '__len__') and not isinstance(v, str) and np.array(beh[k]).ndim == 1]

# Group keys by the length of their arrays
from collections import defaultdict
length_groups = defaultdict(list)
for k in array_keys_1d:
    length_groups[len(beh[k])].append(k)

# Create a DataFrame for each group of arrays with the same length
# Then concatenate them (with different columns) into a single DataFrame, aligning by index
dfs = []
for length, keys in length_groups.items():
    data = {k: np.array(beh[k]) for k in keys}
    df = pd.DataFrame(data)
    dfs.append(df)
    print(f"DataFrame for 1D arrays of length {length}:")
    print(df.head(), "\n")

# Concatenate all DataFrames along columns, aligning by index (outer join)
if dfs:
    beh_df = pd.concat(dfs, axis=1)
    print("Final concatenated DataFrame of all 1D arrays:")
    print(beh_df.head())
else:
    print("No 1D array-like items found in beh.")

DataFrame for 1D arrays of length 348:
   trInd  trInd_odd  trInd_even  Trial_start_time  Trial_end_time  \
0      0       True       False     738235.709693   738235.709813   
1      1      False        True     738235.709813   738235.710091   
2      2       True       False     738235.710091   738235.710468   
3      3      False        True     738235.710468   738235.711075   
4      4       True       False     738235.711075   738235.712070   

   Gray_space_time   SoundPos      SoundTime  SoundTimeDelay        RewTime  \
0    738235.709775  15.200000  738235.709727   738235.709727  738235.709727   
1    738235.709875  12.600000  738235.709830   738235.709830  738235.709830   
2    738235.710189  11.579226  738235.710130   738235.710130  738235.710130   
3    738235.711010  26.100000  738235.710519   738235.710519  738235.710519   
4    738235.712018  15.892707  738235.711128   738235.711128  738235.711128   

   ...  WallIsProbe  WallName    RewardFr     StartFr      GrayFr      

In [4]:
# Information for each trial in contained here 
dfs[0]

,trInd,trInd_odd,trInd_even,Trial_start_time,Trial_end_time,Gray_space_time,SoundPos,SoundTime,SoundTimeDelay,RewTime,...,WallIsProbe,WallName,RewardFr,StartFr,GrayFr,EndFr,SoundFr,SoundDelayFr,SoundDelPos,TrialStim
0,0,True,False,738235.709693,738235.709813,738235.709775,15.200000,738235.709727,738235.709727,738235.709727,...,0.0,leaf1,17.474920,7.830826,30.508013,40.887748,17.474920,17.474920,15.388840,leaf1
1,1,False,True,738235.709813,738235.710091,738235.709875,12.600000,738235.709830,738235.709830,738235.709830,...,0.0,leaf1,45.734435,40.903454,57.904722,117.333398,45.734435,45.734435,72.776751,leaf1
2,2,True,False,738235.710091,738235.710468,738235.710189,11.579226,738235.710130,738235.710130,738235.710130,...,0.0,circle1,NaN,117.397555,144.190894,220.713056,128.000000,128.000000,131.579226,circle1
3,3,False,True,738235.710468,738235.711075,738235.711010,26.100000,738235.710519,738235.710519,738235.710519,...,0.0,leaf1,234.805274,220.774421,369.684923,387.508513,234.805274,234.805274,206.290194,leaf1
4,4,True,False,738235.711075,738235.712070,738235.712018,15.892707,738235.711128,738235.711128,738235.711128,...,0.0,circle1,NaN,387.568069,646.233707,660.677915,402.000000,402.000000,255.892707,circle1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,343,False,True,738235.796028,738235.796314,738235.796276,24.300000,738235.796076,738235.796076,738235.796076,...,0.0,leaf1,23714.765620,23701.535618,23769.693920,23780.079461,23714.765620,23714.765620,20604.522974,leaf1
344,344,True,False,738235.796314,738235.796601,738235.796563,26.400000,738235.796365,738235.796365,738235.796365,...,0.0,leaf1,23794.087492,23780.132707,23848.215516,23858.630153,23794.087492,23794.087492,20666.431759,leaf1
345,345,False,True,738235.796601,738235.797005,738235.796685,20.600000,738235.796641,738235.796641,738235.796641,...,0.0,leaf1,23869.647727,23858.687620,23881.685537,23969.734247,23869.647727,23869.647727,20720.767066,leaf1
346,346,True,False,738235.797006,738235.797256,738235.797211,14.900000,738235.797040,738235.797040,738235.797040,...,0.0,leaf1,23979.116972,23969.797728,24026.262834,24038.544221,23979.116972,23979.116972,20775.014119,leaf1


In [5]:
# Behavioral information for the entire recording is here
dfs[4]

,ft,ft_trInd,ft_trInd_odd,ft_trInd_even,ft_PosCum,ft_Pos,ft_move,ft_isMoving,ft_GraySpc,ft_CorrSpc,ft_WallID,ft_RunCum,ft_RunSpeed,RunFr,BefCueFr,AftCueFr
0,738235.709664,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False
1,738235.709667,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False
2,738235.709671,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False
3,738235.709675,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False
4,738235.709678,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24295,738235.798191,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169491.487206,52.156577,16.824702,False,False
24296,738235.798194,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169508.244950,51.949007,16.757744,False,False
24297,738235.798198,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169525.021443,52.007127,16.776492,False,False
24298,738235.798207,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169568.361563,134.354373,43.340120,False,False


In [6]:
dfs[4][dfs[4]['ft_trInd']==1]

,ft,ft_trInd,ft_trInd_odd,ft_trInd_even,ft_PosCum,ft_Pos,ft_move,ft_isMoving,ft_GraySpc,ft_CorrSpc,ft_WallID,ft_RunCum,ft_RunSpeed,RunFr,BefCueFr,AftCueFr
41,738235.709813,1.0,False,True,60.123507,0.123507,1.780117,True,False,True,leaf1,295.000463,35.015634,11.295366,True,False
42,738235.709817,1.0,False,True,62.497204,2.497204,2.373697,True,False,True,leaf1,310.923899,49.362652,15.923436,True,False
43,738235.709820,1.0,False,True,65.286475,5.286475,2.789271,True,False,True,leaf1,328.766442,55.311883,17.842543,True,False
44,738235.709824,1.0,False,True,67.964429,7.964429,2.677955,True,False,True,leaf1,348.375431,60.787864,19.608988,True,False
45,738235.709828,1.0,False,True,70.734154,10.734154,2.769724,True,False,True,leaf1,367.387010,58.935894,19.011579,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,738235.710075,1.0,False,True,111.605642,51.605642,1.902437,True,True,False,leaf1,518.873440,15.331902,4.945775,False,True
114,738235.710079,1.0,False,True,113.471580,53.471580,1.865938,True,True,False,leaf1,522.918921,12.540993,4.045482,False,True
115,738235.710083,1.0,False,True,115.446064,55.446064,1.974484,True,True,False,leaf1,532.578260,29.943951,9.659339,False,True
116,738235.710086,1.0,False,True,117.235113,57.235113,1.789049,True,True,False,leaf1,546.809094,44.115585,14.230834,False,True


## Timeindex to datetime and to seconds

In [7]:
# Assuming beh_ts_df is your DataFrame
beh_ts_df = dfs[4] 

# The number of days between MATLAB's epoch and Unix's epoch
MATLAB_EPOCH_OFFSET = 719529

# 1. Subtract the offset and specify the unit as 'd' for days
beh_ts_df['datetime'] = pd.to_datetime(beh_ts_df['ft'] - MATLAB_EPOCH_OFFSET, unit='d')

# 2. Now, calculate the total seconds from the start (this part was correct)
beh_ts_df['time'] = (beh_ts_df['datetime'] - beh_ts_df['datetime'].iloc[0]).dt.total_seconds()

beh_ts_df

,ft,ft_trInd,ft_trInd_odd,ft_trInd_even,ft_PosCum,ft_Pos,ft_move,ft_isMoving,ft_GraySpc,ft_CorrSpc,ft_WallID,ft_RunCum,ft_RunSpeed,RunFr,BefCueFr,AftCueFr,datetime,time
0,738235.709664,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False,2021-03-20 17:01:54.948806535,0.000000
1,738235.709667,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False,2021-03-20 17:01:55.262192485,0.313386
2,738235.709671,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False,2021-03-20 17:01:55.576835719,0.628029
3,738235.709675,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False,2021-03-20 17:01:55.891318025,0.942511
4,738235.709678,NaN,False,False,0.000000,0.000000,0.0,False,False,True,nan,-0.000000,0.000000,0.000000,False,False,2021-03-20 17:01:56.206796090,1.257990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24295,738235.798191,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169491.487206,52.156577,16.824702,False,False,2021-03-20 19:09:23.667518643,7648.718712
24296,738235.798194,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169508.244950,51.949007,16.757744,False,False,2021-03-20 19:09:23.982181992,7649.033375
24297,738235.798198,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169525.021443,52.007127,16.776492,False,False,2021-03-20 19:09:24.297197385,7649.348391
24298,738235.798207,NaN,False,False,20879.920052,59.920052,0.0,False,True,False,nan,169568.361563,134.354373,43.340120,False,False,2021-03-20 19:09:25.111003072,7650.162197


In [ ]:
# Begin block: Get info for each trial
beh_trialinfo_df = dfs[0]
beh_trialinfo_df

We need to merge the long behavioral `beh_ts_df` with the trial information `beh_trialinfo_df` 

In [ ]:
# --- Begin block: Assign trial info to each timestamp ---
# Create interval index
intervals = pd.IntervalIndex.from_arrays(beh_trialinfo_df['Trial_start_time'], beh_trialinfo_df['Trial_end_time'], closed='both')

# Get index of intervals matching each ft timestamp
trial_idx = intervals.get_indexer(beh_ts_df['ft'])
trial_idx = intervals.get_indexer(beh_ts_df['ft'])

# Reset index to align indices
beh_trialinfo_df = beh_trialinfo_df.reset_index(drop=True)

# Columns to merge (excluding start/end time if you want)
cols_to_merge = [col for col in beh_trialinfo_df.columns]# if col not in ['Trial_start_time', 'Trial_end_time']]

# Initialize columns in beh_ts_df only once, NOT in beh_trialinfo_df!
for col in cols_to_merge:
    beh_ts_df[col] = None

# For rows where trial_idx != -1, assign corresponding values
mask = trial_idx != -1

for col in cols_to_merge:
    beh_ts_df.loc[mask, col] = beh_trialinfo_df.loc[trial_idx[mask], col].values

In [ ]:
beh_ts_df

# Merge behavioral with spike data

In [ ]:
spikes_df = pd.DataFrame(spikes.T)
spikes_df

In [ ]:
# Check initial offset by comparing the index values of small subset
offset = beh_ts_df.index[0] - spikes_df.index[0]
print(f'Detected offset between index: {offset}')

# Shift spikes_df index by offset
spikes_df_aligned = spikes_df.copy()
spikes_df_aligned.index = spikes_df_aligned.index + offset

# Now join on index
merged_df = beh_ts_df.join(spikes_df_aligned, how='inner')  # or 'left' if you want all beh_ts rows

In [ ]:
merged_df.head(100)

In [ ]:
# choose a trial to visualize
# trial_num = 347
# Select all columns except those in spike_cols
# merged_df.loc[merged_df['ft_trInd'] == trial_num, [col for col in merged_df.columns if col not in spike_cols]][['ft','time', 'SoundPos', 'SoundTime']].head(20)

In [ ]:
# Define the spike_cols names
spike_cols = spikes_df.columns
# Choose a trial to visualize
trial_num = 100
# Select the trial of interest
trial_df = merged_df[merged_df['ft_trInd'] == trial_num][spike_cols]#.drop(columns=spike_cols)

trial_df

In [ ]:
def plot_spike_activity_for_trials(trial_nums, merged_df, spikes_df, figsize_per_plot=(5, 3)):
    """
    Plots spike activity for each trial in trial_nums as a subplot.

    Parameters:
    - trial_nums: list of int, trial numbers to plot
    - merged_df: DataFrame, merged behavioral and spike data
    - spikes_df: DataFrame, original spike data (for column names)
    - figsize_per_plot: tuple, size of each subplot (default (8,8))
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib.lines import Line2D
    

    spike_cols = spikes_df.columns
    n_trials = len(trial_nums)
    ncols = min(n_trials, 2)
    nrows = int(np.ceil(n_trials / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(figsize_per_plot[0]*ncols, figsize_per_plot[1]*nrows), sharex=True, sharey=True)
    if n_trials == 1:
        axes = np.array([[axes]])
    elif nrows == 1 or ncols == 1:
        axes = np.atleast_2d(axes)
    axes = axes.flatten()

    ims = []
    # For legend handles/labels
    legend_handles = []
    legend_labels = []
    legend_added = False

    for idx, trial_num in enumerate(trial_nums):
        ax = axes[idx]
        trial_mask = merged_df['ft_trInd'] == trial_num
        trial_df = merged_df[trial_mask][spike_cols]
        if trial_df.empty:
            ax.set_title(f"Trial # {trial_num} (no data)")
            ax.axis('off')
            ims.append(None)
            continue

        # X-axis: position in meters
        pos_min = merged_df[trial_mask]['ft_Pos'].min() / 10
        pos_max = merged_df[trial_mask]['ft_Pos'].max() / 10

        # To reduce the number of color steps in the colormap, use a ListedColormap with fewer discrete colors.
        import matplotlib as mpl
        from matplotlib.colors import ListedColormap

        # Choose number of discrete color steps
        # n_colors = 12  # for example, 8 color steps
        # base_cmap = plt.get_cmap('Greys')
        # discrete_cmap = ListedColormap(base_cmap(np.linspace(0, 1, n_colors)))

        im = ax.imshow(
            trial_df.T,
            aspect='auto',
            interpolation='nearest',
            cmap='viridis',
            extent=[
                pos_min,
                pos_max,
                0, trial_df.shape[1]
            ],
            # origin='lower'
        )
        ims.append(im)
        ax.set_xlabel('Position (m)')
        ax.set_ylabel('Neuron spike)')
        ax.set_title(f'Spike activity for trial # {trial_num}')
        # ax.grid(False)

        # Add vertical line at SoundPos (convert to float if needed)
        soundpos_handle = None
        gray_handle = None
        try:
            sound_pos = merged_df[trial_mask]['SoundPos'].iloc[0] / 10
            sound_pos = float(sound_pos)
            soundpos_handle = ax.axvline(x=sound_pos, color='red', linestyle='--', label='SoundPos')
        except Exception:
            pass
        gray_handle = ax.axvline(x=4, color='gray', linestyle='--', label='Gray space')

        # Only collect legend handles/labels from the first subplot with data
        if not legend_added:
            handles = []
            labels = []
            if soundpos_handle is not None:
                handles.append(Line2D([0], [0], color='red', linestyle='--'))
                labels.append('SoundPos')
            handles.append(Line2D([0], [0], color='gray', linestyle='--'))
            labels.append('Gray space')
            legend_handles = handles
            legend_labels = labels
            legend_added = True

    # Hide any unused subplots
    for j in range(idx+1, len(axes)):
        axes[j].axis('off')

    # Add a single colorbar to the right of all subplots
    # Find the first valid imshow object for the colorbar
    for im in ims:
        if im is not None:
            im_for_cbar = im
            break
    else:
        im_for_cbar = None

    if im_for_cbar is not None:
        # Place the colorbar to the right of all subplots
        fig.subplots_adjust(right=0.80)
        cbar_ax = fig.add_axes([0.82, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
        fig.colorbar(im_for_cbar, cax=cbar_ax, orientation='vertical', label='Spike value')

    # Add a single legend to the top right of all subplots
    if legend_handles and legend_labels:
        # Place the legend at the top right
        fig.legend(
            handles=legend_handles,
            labels=legend_labels,
            loc='upper right',
            bbox_to_anchor=(0.98, 0.98),
            borderaxespad=0.2,
            frameon=True,
            title="Legend"
        )

    plt.tight_layout(rect=[0, 0, 0.80, 1])
    
    plt.show()

In [ ]:
# Example usage:
trials_to_plot = [1, 50, 100, 150, 229, 347]
plot_spike_activity_for_trials(trials_to_plot, merged_df, spikes_df)

The code below is for testing purposes, not intended to be run.

In [ ]:
# Visualize the column names in merged_df that match the spikes_df columns, for each trial (trInd)

# Get the spike columns (these are the columns from spikes_df, which are numeric and not in beh_ts_df)
# spike_cols = [col for col in spikes_df.columns if col in merged_df.columns]

# # For each unique trial index (trInd), print the trial and the spike columns present
# if 'ft_trInd' in merged_df.columns:
#     for tr in merged_df['ft_trInd'].dropna().unique():
#         print(f"Trial {tr}:")
#         trial_df = merged_df[merged_df['ft_trInd'] == tr]
#         # Show the spike columns present (should be all, but just in case)
#         present_spike_cols = [col for col in spike_cols if col in trial_df.columns]
#         print(present_spike_cols)
#         print('-' * 40)
# else:
#     print("Column 'ft_trInd' not found in merged_df.")

# # Optionally, visualize as a heatmap (trial x neuron) of spike activity
# import matplotlib.pyplot as plt
# import numpy as np

# if 'ft_trInd' in merged_df.columns:
#     # Group by trial and take mean spike activity per neuron per trial
#     trial_spike_means = merged_df.groupby('ft_trInd')[spike_cols].mean()
#     plt.figure(figsize=(12,6))
#     plt.imshow(trial_spike_means, aspect='auto', interpolation='nearest', cmap='viridis')
#     plt.colorbar(label='Mean spike activity')
#     plt.xlabel('Neuron (spike column)')
#     plt.ylabel('Trial (ft_trInd)')
#     plt.title('Mean spike activity per neuron per trial')
#     plt.show()

# Constructing PCA matrix
- We calculate each neuron's mean activation from entering the tunnel until the gray area
    -  Per stimilus condition, before and after (for both, supervised and unsupervised learning)


In [ ]:
n_trials_leaf1 = merged_df[(merged_df['ft_Pos'] < 40) & (merged_df['TrialStim'] == 'leaf1')]['ft_trInd'].nunique()
n_trials_circle1 = merged_df[(merged_df['ft_Pos'] < 40) & (merged_df['TrialStim'] == 'circle1')]['ft_trInd'].nunique()
print(f"Number of trials for leaf1: {n_trials_leaf1}")
print(f"Number of trials for circle1: {n_trials_circle1}")

In [ ]:
def generate_trial_averaged_spiking_data(data, spike_cols, trialStim):
    """
    Returns a (n_neurons, n_trials) array of trial-averaged spiking data for a given stimulus.
    Each column is the mean activity for all neurons in one trial.
    The columns are named using the first letter of trialStim and the trial number (e.g., "l_0", "l_1", ...).
    """
    trial_col_name = 'ft_trInd'  # trial column name
    # Filter data for the given stimulus and position
    data = data[(data['ft_Pos'] < 40) & (data['TrialStim'] == trialStim)]
    # Get unique trial numbers (sorted for consistency)
    trial_numbers = np.sort(data[trial_col_name].dropna().unique().astype(int))
    n_neurons = len(spike_cols)
    n_trials = len(trial_numbers)
    trial_averaged_spiking_data = np.empty((n_neurons, n_trials))
    for idx, trial in enumerate(trial_numbers):
        trial_data = data[data[trial_col_name] == trial][spike_cols]
        trial_avg = trial_data.mean(axis=0).to_numpy()
        trial_averaged_spiking_data[:, idx] = trial_avg
    # Create column names with only the first letter of trialStim and trial number
    col_names = [f"{trialStim[0]}_{trial}" for trial in trial_numbers]
    return pd.DataFrame(trial_averaged_spiking_data, columns=col_names)

In [ ]:
spike_cols = spikes_df.columns
trial_means_leaf1 = generate_trial_averaged_spiking_data(merged_df, spike_cols, trialStim='leaf1')
trial_means_circle1 = generate_trial_averaged_spiking_data(merged_df, spike_cols, trialStim='circle1')

In [ ]:
# Mean activation per neuron (rows) x trials (columns)
trial_means_circle1

In [ ]:
# Mean activation per neuron (rows) x trials (columns)
trial_means_leaf1

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
import re

def extract_trial_number(colname):
    # Extract the number after the underscore
    match = re.search(r'_(\d+)', colname)
    return int(match.group(1)) if match else None

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
fig.suptitle('Trial-averaged spiking per stimulus condition')

# For leaf1
im0 = axes[0].imshow(
    trial_means_leaf1,
    aspect='auto',
    interpolation='nearest',
    cmap='viridis',
)
axes[0].set_title('Condition: leaf1')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Neurons')

# Set x-ticks every 10th trial
n_cols_leaf1 = len(trial_means_leaf1.columns)
tick_indices_leaf1 = list(range(0, n_cols_leaf1, 10))
tick_labels_leaf1 = [
    str(extract_trial_number(trial_means_leaf1.columns[i])) for i in tick_indices_leaf1
]
axes[0].set_xticks(tick_indices_leaf1)
axes[0].set_xticklabels(tick_labels_leaf1, rotation=90, fontsize=8)

# For circle1
im1 = axes[1].imshow(
    trial_means_circle1,
    aspect='auto',
    interpolation='nearest',
    cmap='viridis',
)
axes[1].set_title('Condition: circle1')
axes[1].set_xlabel('Trial')

n_cols_circle1 = len(trial_means_circle1.columns)
tick_indices_circle1 = list(range(0, n_cols_circle1, 10))
tick_labels_circle1 = [
    str(extract_trial_number(trial_means_circle1.columns[i])) for i in tick_indices_circle1
]
axes[1].set_xticks(tick_indices_circle1)
axes[1].set_xticklabels(tick_labels_circle1, rotation=90, fontsize=8)

# Add a colorbar to the right of the second subplot without shrinking the axes
divider = make_axes_locatable(axes[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = fig.colorbar(im1, cax=cax, orientation='vertical', label='Spike value')

plt.tight_layout()
plt.show()

In [ ]:
trial_means_leaf1.T

In [ ]:
# Generate trial-averaged spiking data for each stimulus
# trial_means_leaf1 = generate_trial_averaged_spiking_data(merged_df, spike_cols, trialStim='leaf1')
# trial_means_circle1 = generate_trial_averaged_spiking_data(merged_df, spike_cols, trialStim='circle1')

# Convert to DataFrames and add 'TrialStim' column
df_leaf1 = trial_means_leaf1.T  # shape: (n_trials, n_neurons)
df_leaf1['TrialStim'] = 'leaf1'
df_leaf1['TrialNum'] = trial_means_leaf1.T.index

df_circle1 = trial_means_circle1.T
df_circle1['TrialStim'] = 'circle1'
df_circle1['TrialNum'] = trial_means_circle1.T.index

# Merge the two DataFrames
trial_means_merged = pd.concat([df_leaf1, df_circle1], ignore_index=False)
trial_means_merged

In [ ]:
# from sklearn.decomposition import PCA
# import matplotlib.pyplot as plt

# # Fit PCA and project the data onto the first two principal components
# pca = PCA(n_components=2)
# projected = pca.fit_transform(trial_means_merged[spike_cols])
# print(pca.components_)
# projected_df = pd.DataFrame(projected)
# projected_df['TrialStim'] = trial_means_merged['TrialStim']
# projected_df['TrialNum'] = trial_means_merged['ft_trInd']
# projected_df

# Calculate d-prime for supervised mouse before learning

In [ ]:
# function for calculating one of the selective index: d-prime
def dprime(spk1, spk2):
    u1 = np.nanmean(spk1, axis=1) # take mean across frames
    u2 = np.nanmean(spk2, axis=1)
    sig1 = np.nanstd(spk1, axis=1)
    sig2 = np.nanstd(spk2, axis=1)
    dp = 2 * (u1 - u2) / (sig1 + sig2) # get d-prime
    return dp

# @title Rewriting code above into a function to get the d-prime for other sessions (mice)
def cal_dprime(root, m_name, exp, stim1_ID=2, stim2_ID=0):
    """m_name: name of recording
       exp: experiment
       stim1_ID and stim2_ID: ID of stimulus 1 and 2, 0:circle1, 1:circle2, 2:leaf1, 3:leaf2, 4:leaf3;
       rewarded stimulus (leaf1) for supervised experiment, non-rewarded stimulus (circle1) for supervised experiment
    """
    # load behavior data
    beh = np.load(os.path.join(root, exp), allow_pickle=1).item()[m_name]
    stim_id = beh['stim_id'] # 0:circle1, 1:circle2, 2:leaf1, 3:leaf2, 4:leaf3
    ntrials, uniqW, WallN, SoundPos = beh['ntrials'], beh['UniqWalls'], beh['WallName'], beh['SoundPos']

    stim_fr = beh['ft_WallID'] # trial stimulus of each frame
    pos_fr = beh['ft_Pos'] # position of each frame inside corridor
    VR_move = beh['ft_move']>0 # frames with virtual scene moving

    name_stim1 = uniqW[stim_id==stim1_ID][0] # name of stimulus 1;
    name_stim2 = uniqW[stim_id==stim2_ID][0] # name of stimulus 2;

    stim1_idx = (stim_fr==name_stim1) & (pos_fr<40)
    stim2_idx = (stim_fr==name_stim2) & (pos_fr<40)

    idx1 = stim1_idx & VR_move # only take frames while mouse is running inside the texture area
    idx2 = stim2_idx & VR_move
    svd_ = np.load(os.path.join(root, m_name + '_SVD_dec.npy'), allow_pickle=1).item()
    proj = svd_['U'].T @ svd_['V'] # project from the PC space back to neural space
    del svd_
    nfrs = proj.shape[1]
    # using the function 'dprime' if you have more RAM, otherwise use a for loop
    dp = dprime(proj[:, idx1[:nfrs]], proj[:, idx2[:nfrs]]) # get d-prime
    # u1, u2, sig1, sig2 = np.empty(proj.shape[0]), np.empty(proj.shape[0]), np.empty(proj.shape[0]), np.empty(proj.shape[0])
    # for n in range(proj.shape[0]):
    #     u1[n] = np.nanmean(proj[n, idx1[:nfrs]])
    #     u2[n] = np.nanmean(proj[n, idx2[:nfrs]])
    #     sig1[n] = np.nanstd(proj[n, idx1[:nfrs]])
    #     sig2[n] = np.nanstd(proj[n, idx2[:nfrs]])
    # dp = 2 * (u1 - u2) / (sig1 + sig2) # get d-prime
    del proj
    retin = np.load(os.path.join(root, m_name[:-1]+'trans.npz'), allow_pickle=1)
    xpos, ypos = retin['xpos']/0.75, retin['ypos']/0.5

    return {'dp':dp, 'xpos':xpos, 'ypos':ypos, 'iarea':retin['iarea']}

In [ ]:
dp_sup_bef = cal_dprime(root, sup_bef, 'Beh_sup_train1_before_learning.npy')
dp_sup_bef_df = pd.DataFrame(dp_sup_bef)
dp_sup_bef_df


## Identify Neural Selectivity and Brain Area

In [ ]:
thr = 0.6  # set d-prime threshold

# Create selectivity column based on dp threshold
def selectivity_label(dp):
    if dp >= thr:
        return 'leaf1'
    elif dp <= -thr:
        return 'circle1'
    else:
        return None
# Map iarea numbers to string names using the same logic as neu_area_ID
def iarea_to_name(iarea_val):
    if iarea_val == 8:
        return 'V1'
    elif iarea_val in [0, 1, 2, 9]:
        return 'mHV'
    elif iarea_val in [5, 6]:
        return 'lHV'
    elif iarea_val in [3, 4]:
        return 'aHV'
    else:
        return 'Other'

# Add a new column for area names
dp_sup_bef_df['area_name'] = dp_sup_bef_df['iarea'].apply(iarea_to_name)
# trial_means_merged = trial_means_merged.copy()
dp_sup_bef_df['selectivity'] = dp_sup_bef_df['dp'].apply(selectivity_label)
dp_sup_bef_df

# Functions for PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

def pca(X):
    """
    Performs PCA on multivariate data. Eigenvalues are sorted in decreasing order.

    Args:
        X (numpy array or pandas DataFrame): Data matrix, each column corresponds to a
                                             different variable (feature/neuron).

    Returns:
        score (numpy array): Data projected onto the new basis (principal components).
        evectors (numpy array): Matrix of eigenvectors (principal axes).
        evals (numpy array): Vector of eigenvalues (explained variances).
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca_model = PCA()# when samples are less thatn variables use svd_solver='randomized'
    score = pca_model.fit_transform(X_scaled)
    evectors = pca_model.components_.T  # shape: features × components
    evals = pca_model.explained_variance_  # eigenvalues of covariance matrix
    return score, evectors, evals

def plot_eigenvalues(evals, xlimit=False, xlim_max = 0):
  """
  Plots eigenvalues.

  Args:
     (numpy array of floats) : Vector of eigenvalues
     (boolean) : enable plt.show()
  Returns:
    Nothing.

  """

  plt.figure()
  # plt.plot(np.arange(1, len(evals) + 1), evals, 'o-k')
  plt.plot(evals, 'o-k')
  plt.xlabel('Component')
  plt.ylabel('Eigenvalue')
  plt.title('Scree plot')
  if xlimit:
    plt.xlim([0, xlim_max])  # limit x-axis up to 100 for zooming
  plt.show()

def get_variance_explained(evals):
    """
    Calculates cumulative variance explained from the eigenvalues.

    Args:
      evals (numpy array of floats): Vector of eigenvalues

    Returns:
      (numpy array of floats): Vector of cumulative variance explained
    """
    # Cumulative sum of eigenvalues
    csum = np.cumsum(evals)
    # Normalize by the total sum of eigenvalues
    variance_explained = csum / np.sum(evals)
    return variance_explained

def plot_variance_explained(variance_explained):
    """
    Plots cumulative explained variance with a horizontal line at 90%
    and a vertical line where it crosses.

    Args:
        variance_explained (numpy array of floats): Cumulative variance explained per PC

    Returns:
        Nothing.
    """
    threshold = 0.9
    x = np.arange(1, len(variance_explained) + 1)

    # Find first index where variance crosses threshold
    idx = np.argmax(variance_explained >= threshold)

    # If threshold never crossed, fallback to last component
    if variance_explained[idx] < threshold:
        intersection_x = x[-1]
    elif idx == 0:
        # threshold crossed at or before first component
        intersection_x = x[0]
    else:
        # Linear interpolation for more accurate crossing point
        x0, x1 = x[idx - 1], x[idx]
        y0, y1 = variance_explained[idx - 1], variance_explained[idx]

        # slope = (y1 - y0) / (x1 - x0)
        # Intersection at y=threshold: x = x0 + (threshold - y0)/slope
        slope = y1 - y0
        intersection_x = x0 + (threshold - y0) / slope

    plt.figure(figsize=(8, 5))
    plt.plot(x, variance_explained, '--k', label='Cumulative Variance')

    plt.axhline(y=threshold, color='red', linestyle='--', label=f'{int(threshold * 100)}% Variance')
    plt.axvline(x=intersection_x, color='green', linestyle='--', label=f'Intersection at {intersection_x:.2f} Components')

    plt.xlabel('Number of components')
    plt.ylabel('Cumulative explained variance')
    plt.title('PCA: Components vs Explained Variance')
    plt.legend()
    plt.show()

# Merge spikes data with D-prime data

In [ ]:
spikes_for_pcs = trial_means_merged[spike_cols].T
spikes_for_pcs

In [ ]:
dp_sup_bef_df

In [ ]:
# Concatenate the DataFrames side by side
trial_means_merged = pd.concat([spikes_for_pcs, dp_sup_bef_df], axis=1)

# Save the merged DataFrame to CSV in the project directory
trial_means_merged.to_csv(root + "trial_means_merged.csv", index=True)

trial_means_merged

## Spike activity by brain area

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Get unique area names (excluding 'Other' if you want)
area_names = [name for name in trial_means_merged['area_name'].unique() if name != 'Other']

# Prepare for plotting: one row per area, two columns (leaf1, circle1)
n_areas = len(area_names)
fig, axes = plt.subplots(n_areas, 2, figsize=(10, 4 * n_areas), sharey=False, sharex=True)
if n_areas == 1:
    axes = axes.reshape(1, 2)  # Ensure axes is always 2D

fig.suptitle('Trial-averaged spiking per stimulus condition by brain area', fontsize=16, y=1.02)

for i, area in enumerate(area_names):
    df_area = trial_means_merged[trial_means_merged['area_name'] == area]
    leaf1_cols = [col for col in df_area.columns if 'l_' in str(col)]
    circle1_cols = [col for col in df_area.columns if 'c_' in str(col)]

    # Plot leaf1
    im0 = axes[i, 0].imshow(
        df_area[leaf1_cols],
        aspect='auto',
        interpolation='nearest',
        cmap='viridis',
    )
    axes[i, 0].set_title(f'{area}: leaf1')
    axes[i, 0].set_xlabel('Trials Number')
    axes[i, 0].set_ylabel('Neurons')

    # Plot circle1
    im1 = axes[i, 1].imshow(
        df_area[circle1_cols],
        aspect='auto',
        interpolation='nearest',
        cmap='viridis',
    )
    axes[i, 1].set_title(f'{area}: circle1')
    axes[i, 1].set_xlabel('Trials Number')

    # Add a colorbar to the right of the second subplot in each row
    divider = make_axes_locatable(axes[i, 1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(im1, cax=cax, orientation='vertical', label='Spike value')

plt.tight_layout()
plt.show()

# Apply PCA

In [ ]:
spikes_for_pcs

In [ ]:
# Perform PCA
score, evectors, evals = pca(spikes_for_pcs)

In [ ]:
# Plot the eigenvalues
plot_eigenvalues(evals, xlimit=False, xlim_max=10)

In [ ]:
# Calculate the variance explained
variance_explained = get_variance_explained(evals)

# Visualize
plot_variance_explained(variance_explained)

In [ ]:
spikes_for_pcs

In [ ]:
# Fit PCA and project the data onto the first three principal components
pca = PCA()
projected = pca.fit_transform(spikes_for_pcs)
evectors = pca.components_.T  # shape: features × components
evals = pca.explained_variance_  # eigenvalues of covariance matrix

projected_df = pd.DataFrame(projected)
projected_df['dp'] = trial_means_merged['dp']
projected_df['iarea'] = trial_means_merged['iarea']
projected_df['selectivity'] = trial_means_merged['selectivity']
# projected_df = pd.DataFrame(projected)

# Calculate the variance explained
variance_explained = get_variance_explained(evals)

# Calculate percentage of explained variance for PC1 and PC2
pc1_var = variance_explained[0] * 100
pc2_var = (variance_explained[1] - variance_explained[0]) * 100

# Create a 2x2 subplot for all visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Top left: Projected onto first two PCs (scatter)
sns.scatterplot(

    data=projected_df, x=projected[:, 0], y=projected[:, 1],
    alpha=0.2, c=projected_df['dp'], cmap='viridis',ax=axes[0, 0], s=20)

axes[0, 0].set_xlabel(f'PC1 ({pc1_var:.1f}% var)')
axes[0, 0].set_ylabel(f'PC2 ({pc2_var:.1f}% var)')
axes[0, 0].set_title('Projected onto first two PCs')

axes[0,0].set_title('Projected onto first two PCs by dp value')
# cbar = plt.colorbar(sc, ax=axes[0,0], label='dp value')
# cbar.set_label('dp value')

# Top right: Visualize variable contributions (loadings) as vectors in PC1-PC2 space
# Each variable is a vector from (0,0) to (loading on PC1, loading on PC2)
axes[0, 1].axhline(0, color='gray', linewidth=0.5)
axes[0, 1].axvline(0, color='gray', linewidth=0.5)
for i in range(evectors.shape[0]):
    axes[0, 1].arrow(0, 0, evectors[i, 0], evectors[i, 1], 
                     color='blue', alpha=0.2, head_width=0.005, linewidth=0.1,
                     head_length=0.005, length_includes_head=True)
# Optionally, label a few top contributing variables
top_n = 2
contrib = np.sqrt(evectors[:, 0]**2 + evectors[:, 1]**2)
top_idx = np.argsort(contrib)[-top_n:]
# Use the feature names from spikes_for_pcs.columns for labeling
feature_names = spikes_for_pcs.columns if hasattr(spikes_for_pcs, 'columns') else [str(i) for i in range(evectors.shape[0])]
for i in top_idx:
    axes[0, 1].text(evectors[i, 0], evectors[i, 1], str(feature_names[i]), 
                    fontsize=6, color='red')
axes[0, 1].set_xlabel('PC1 loading')
axes[0, 1].set_ylabel('PC2 loading')
axes[0, 1].set_title(f'Top {top_n} Variable Contributions to PC1 and PC2')
axes[0, 1].set_aspect('equal')
axes[0, 1].set_xlim(-0.05, 0.15)
axes[0, 1].set_ylim(-0.15, 0.12)

# Bottom left: Variance explained plot
axes[1, 0].plot(np.arange(1, len(variance_explained)+1), variance_explained, marker='o')
axes[1, 0].set_xlabel('Number of Principal Components')
axes[1, 0].set_ylabel('Cumulative Explained Variance')
axes[1, 0].set_title('Cumulative Variance Explained')

# Bottom right: Eigenvalues plot
axes[1, 1].plot(np.arange(1, len(evals)+1), evals, marker='o')
axes[1, 1].set_xlabel('Principal Component')
axes[1, 1].set_ylabel('Eigenvalue')
axes[1, 1].set_title('Eigenvalues of Covariance Matrix')
# plt.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
spikes_for_pcs


In [ ]:
# numpy as np
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt
# import pandas as pd
import seaborn as sns

# Fit PCA and project the data onto the first three principal components
pca = PCA()
projected = pca.fit_transform(spikes_for_pcs)
evectors = pca.components_.T  # shape: features × components
evals = pca.explained_variance_  # eigenvalues of covariance matrix

projected_df = pd.DataFrame(projected)
projected_df['dp'] = trial_means_merged['dp']
projected_df['iarea'] = trial_means_merged['iarea']
projected_df['selectivity'] = trial_means_merged['selectivity']
# projected_df = pd.DataFrame(projected)

# Calculate the variance explained
variance_explained = get_variance_explained(evals)

# Calculate percentage of explained variance for PC1 and PC2
pc1_var = variance_explained[0] * 100
pc2_var = (variance_explained[1] - variance_explained[0]) * 100

# Create a 1x3 subplot for all visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Add panel labels (A, B, C) to each subplot in the upper left corner
panel_labels = ['(A)', '(B)', '(C)']
axes_flat = axes.flatten()
for i, ax in enumerate(axes_flat):
    # Adjust x position for left column subplots for better alignment
    if i % 2 == 0:  # left column
        x_pos = -0.15
    else:           # right column
        x_pos = -0.3
    ax.text(
        x_pos, 1.1, panel_labels[i], transform=ax.transAxes,
        fontsize=18, fontweight='bold', va='top', ha='left'
    )

# Left: Projected onto first two PCs by dp value (TrialStim) with continuous colormap
sc = axes[0].scatter(
    projected[:, 0], projected[:, 1],
    c=projected_df['dp'], cmap='viridis', alpha=0.7, s=10, vmin=-1.6, vmax=1.6
)
axes[0].set_xlabel(f'PC1 ({pc1_var:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pc2_var:.1f}% var)')
axes[0].set_title('Projected onto first two PCs by dp value')
cbar = plt.colorbar(sc, ax=axes[0], label='dp value')
cbar.set_label('d-prime value')

# Middle: Visualize variable contributions (loadings) as vectors in PC1-PC2 space
# Each variable is a vector from (0,0) to (loading on PC1, loading on PC2)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].axvline(0, color='gray', linewidth=0.5)
for i in range(evectors.shape[0]):
    axes[1].arrow(0, 0, evectors[i, 0], evectors[i, 1], 
                  color='blue', alpha=0.2, head_width=0.005, linewidth=0.1,
                  head_length=0.005, length_includes_head=True)
# Optionally, label a few top contributing variables
top_n = 50
contrib = np.sqrt(evectors[:, 0]**2 + evectors[:, 1]**2)
top_idx = np.argsort(contrib)[-top_n:]
# Use the feature names from spikes_for_pcs.columns for labeling
feature_names = spikes_for_pcs.columns if hasattr(spikes_for_pcs, 'columns') else [str(i) for i in range(evectors.shape[0])]
for i in top_idx:
    axes[1].text(evectors[i, 0], evectors[i, 1], str(feature_names[i]), 
                 fontsize=6, color='red')
axes[1].set_xlabel('PC1 loading')
axes[1].set_ylabel('PC2 loading')
axes[1].set_title(f'Top {top_n} Trials Contributions')
axes[1].set_aspect('equal')
axes[1].set_xlim(-0.05, 0.15)
axes[1].set_ylim(-0.15, 0.12)

# Right: Projected onto first two PCs by selectivity (leaf1/circle1/None) with colorblind-friendly palette
# Define a colorblind-friendly palette for selectivity
selectivity_palette = {
    'leaf1': '#2F9C92',    # blue
    'circle1': '#E4C16E',  # vermillion
}
# Ensure the order for legend
selectivity_order = ['leaf1', 'circle1']

sns.scatterplot(
    data=projected_df, x=projected[:, 0], y=projected[:, 1],
    alpha=1, ax=axes[2], hue='selectivity', legend=True,
    palette=selectivity_palette, hue_order=selectivity_order
)
axes[2].set_xlabel(f'PC1 ({pc1_var:.1f}% var)')
axes[2].set_ylabel(f'PC2 ({pc2_var:.1f}% var)')
axes[2].set_title('Projected onto first two PCs by selectivity')
axes[2].legend(title='Selectivity', bbox_to_anchor=(1, 1), loc='upper left')

plt.tight_layout()
plt.savefig('results/pca_supervised_selectivity_contributions.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
import re
from matplotlib.patches import Patch

pca = PCA()
projected = pca.fit_transform(spikes_for_pcs)
# Visualize Eigenvectors
eigenvectors = pca.components_.T  # Shape: (n_features, n_components)

# Extract feature names from spikes_for_pcs
feature_names = spikes_for_pcs.columns if hasattr(spikes_for_pcs, 'columns') else [str(i) for i in range(eigenvectors.shape[0])]

# Parse condition and trial number from feature names
conditions = []
trial_numbers = []
for name in feature_names:
    m = re.match(r'([lc])_(\d+)', str(name))
    if m:
        cond = m.group(1)
        trial = int(m.group(2))
    else:
        # cond = 'other'
        trial = name
    conditions.append(cond)
    trial_numbers.append(trial)

# Sort by trial_numbers (x-axis) in ascending order
sort_idx = sorted(range(len(trial_numbers)), key=lambda i: trial_numbers[i])
sorted_trial_numbers = [trial_numbers[i] for i in sort_idx]
sorted_conditions = [conditions[i] for i in sort_idx]
sorted_eigenvectors = eigenvectors[sort_idx, :]

# Assign a color to each condition
condition_palette = {'l': '#2F9C92', 'c': '#E4C16E'}
sorted_colors = [condition_palette.get(cond, '#999999') for cond in sorted_conditions]

# Only label every 20th tick on the x-axis
xtick_locs = [i for i in range(len(sorted_trial_numbers)) if i % 20 == 0]
xtick_labels = [sorted_trial_numbers[i] for i in xtick_locs]

# Create a 1 row, 2 columns subplot for both bar plots
fig, axes = plt.subplots(1, 2, figsize=(18, 4), sharex=True)
# Add (A), (B), (C) labels above each subplot
panel_labels = ['(D)', '(F)']
axes_flat = axes.flatten()
for i, ax in enumerate(axes_flat):
    # Adjust x position for left column subplots for better alignment
    ax.text(
        -0.1, 1.1, panel_labels[i], transform=ax.transAxes,
        fontsize=18, fontweight='bold', va='top', ha='left'
    )
# Bar plot for PC1 loadings
bars1 = axes[0].bar(range(len(sorted_trial_numbers)), sorted_eigenvectors[:, 0], color=sorted_colors)
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('PC1 Loading')
axes[0].set_title('PC1 Contributions by Trial Number')
axes[0].set_xticks(xtick_locs)
axes[0].set_xticklabels(xtick_labels, rotation=0, fontsize=10)
axes[0].set_xlim(-2, 347)

# Add legend for conditions (only once)
legend_elements = [
    Patch(facecolor=condition_palette['l'], label='leaf'),
    Patch(facecolor=condition_palette['c'], label='circle'),
    # Patch(facecolor=condition_palette['other'], label='other')
]
axes[0].legend(handles=legend_elements, title='Condition', loc='upper right')

# Bar plot for PC2 loadings
bars2 = axes[1].bar(range(len(sorted_trial_numbers)), sorted_eigenvectors[:, 1], color=sorted_colors)
axes[1].set_xlabel('Trial Number')
axes[1].set_ylabel('PC2 Loading')
axes[1].set_title('PC2 Contributions by Trial Number')
axes[1].set_xticks(xtick_locs)
axes[1].set_xlim(-2, 347)
axes[1].set_xticklabels(xtick_labels, rotation=0, fontsize=10)
axes[1].legend(handles=legend_elements, title='Condition', loc='upper right')

plt.tight_layout()
plt.savefig('results/pca_supervised_trial_contributions_pc1_pc2.png', dpi=300, bbox_inches='tight', transparent=True)

plt.show()

In [ ]:
trial_means_merged[trial_means_merged['selectivity'].notna()]

In [ ]:
# Fit PCA and project the data onto the first three principal components
pca = PCA()
projected = pca.fit_transform(spikes_for_pcs)
evectors = pca.components_.T  # shape: features × components
evals = pca.explained_variance_  # eigenvalues of covariance matrix

projected_df = pd.DataFrame(projected)
projected_df['dp'] = trial_means_merged['dp']
projected_df['iarea'] = trial_means_merged['iarea']
projected_df['selectivity'] = trial_means_merged['selectivity']

# Map iarea numbers to string names using the same logic as neu_area_ID
def iarea_to_name(iarea_val):
    if iarea_val == 8:
        return 'V1'
    elif iarea_val in [0, 1, 2, 9]:
        return 'mHV'
    elif iarea_val in [5, 6]:
        return 'lHV'
    elif iarea_val in [3, 4]:
        return 'aHV'
    else:
        return 'Other'

projected_df['iarea_name'] = projected_df['iarea'].apply(iarea_to_name)

# Calculate the variance explained
variance_explained = get_variance_explained(evals)

# Calculate percentage of explained variance for PC1 and PC2
pc1_var = variance_explained[0] * 100
pc2_var = (variance_explained[1] - variance_explained[0]) * 100

# Create a 2x1 subplot for both visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Left: Projected onto first two PCs by dp value (TrialStim) with continuous colormap
sc = axes[0].scatter(
    projected[:, 0], projected[:, 1],
    c=projected_df['dp'], cmap='viridis', alpha=0.7, s=10, vmin=-1.6, vmax=1.6
)
axes[0].set_xlabel(f'PC1 ({pc1_var:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pc2_var:.1f}% var)')
axes[0].set_title('Projected onto first two PCs by dp value')
cbar = plt.colorbar(sc, ax=axes[0], label='dp value')
cbar.set_label('dp value')

# Right: Projected onto first two PCs by selectivity (leaf1/circle1/None) with colorblind-friendly palette
# Define a colorblind-friendly palette for selectivity
selectivity_palette = {
    'leaf1': '#2F9C92',    # blue
    'circle1': '#E4C16E',  # vermillion
    None: '#999999'        # grey for None
}
# Ensure the order for legend
selectivity_order = ['leaf1', 'circle1', None]

sns.scatterplot(
    data=projected_df, x=projected[:, 0], y=projected[:, 1],
    alpha=0.5, ax=axes[1], hue='selectivity', legend=True,
    palette=selectivity_palette, hue_order=selectivity_order
)
axes[1].set_xlabel(f'PC1 ({pc1_var:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pc2_var:.1f}% var)')
axes[1].set_title('Projected onto first two PCs by selectivity')
axes[1].legend(title='Selectivity', bbox_to_anchor=(1.05, 1), loc='upper left')

# Bottom: Projected onto first two PCs by brain area (iarea_name) with categorical colormap
iarea_sel = projected_df[projected_df['selectivity'].notna()]

# Create a figure and axis
fig, ax = plt.subplots(figsize=(14, 7))

# Use seaborn scatterplot for categorical coloring by iarea_name
sns.scatterplot(
    data=iarea_sel,
    x=0, y=1,
    hue='iarea_name',
    palette='tab10',
    alpha=0.7,
    s=50,
    ax=ax,
    legend='full'
)
ax.set_xlabel(f'PC1 ({pc1_var:.1f}% var)')
ax.set_ylabel(f'PC2 ({pc2_var:.1f}% var)')
ax.set_title('Projected onto first two PCs by iarea')
ax.legend(title='iarea name', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
spikes_for_pcs

## PCA Selective Neurons Subset

In [ ]:
# Filter for selectivity and then sort by 'area_name' so similar strings are grouped together
selective_df = trial_means_merged[trial_means_merged['selectivity'].notna()]
# selective_df = trial_means_merged[(trial_means_merged['selectivity'].notna()) & (trial_means_merged['area_name'] == 'lHV')]

In [ ]:
selective_df

In [ ]:
# from scipy.stats.mstats import winsorize
# import numpy as np
# import pandas as pd

# # Convert the DataFrame to a NumPy array
# data_array = trial_means_merged[spike_cols].to_numpy()

# # Apply winsorizing to the NumPy array
# winsorized_data = winsorize(data_array, limits=[0.05, 0.05])  # 5% winsorization

# # Convert back to DataFrame (optional, depending on downstream analysis)
# winsorized_df = pd.DataFrame(winsorized_data, columns=trial_means_merged[spike_cols].columns)

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt
# import seaborn as sns

# --- 1. Create Sample Data ---
# Since we don't have your original 'selective_df', let's create a sample
# DataFrame that mimics its structure. This makes the script runnable.
# We'll create some correlated features to make the PCA interesting.
# np.random.seed(42)
# # Define a covariance matrix to create correlated variables
# cov_matrix = np.array([
#     [2.0, 1.5, 0.8, 0.5, -0.5],
#     [1.5, 2.5, 1.0, 0.6, -0.7],
#     [0.8, 1.0, 1.5, 0.9, -0.4],
#     [0.5, 0.6, 0.9, 1.2, -0.3],
#     [-0.5, -0.7, -0.4, -0.3, 1.8]
# ])
# # Generate data from a multivariate normal distribution
# data = np.random.multivariate_normal(mean=[0, 0, 0, 0, 0], cov=cov_matrix, size=200)
# feature_names_sample = ['Feature A', 'Feature B', 'Feature C', 'Feature D', 'Feature E']
# selective_df = pd.DataFrame(data, columns=feature_names_sample)

# # Add the other columns you were dropping, so the script matches yours
# selective_df['dp'] = np.random.rand(200)
# selective_df['xpos'] = np.random.rand(200)
# selective_df['ypos'] = np.random.rand(200)
# selective_df['iarea'] = np.random.rand(200)
# selective_df['selectivity'] = np.random.rand(200)


# --- 2. Data Preprocessing and PCA (Your original logic) ---
# Select the features to use for PCA
# features_for_pca = selective_df[spikes_for_pcs.columns]#.drop(columns=['dp', 'xpos', 'ypos','area_name', 'iarea', 'selectivity'], inplace=False)

# # Scale the data
# scaler = StandardScaler()
# data_scaled = scaler.fit_transform(features_for_pca.T)

# # Perform PCA
# n_components = 2
# pca = PCA(n_components=n_components)
# scores = pca.fit_transform(data_scaled)
# scores_df = pd.DataFrame(scores, columns=['PC1', 'PC2'])

# # Get the loadings (eigenvectors)
# loadings = pca.components_.T  # shape: (n_features, n_components)
# feature_names = pd.DataFrame(data_scaled).columns

# # 8. Add the selectivity column to the dataframe
# scores_df['selectivity'] = selective_df['selectivity'].reset_index(drop=True)
# scores_df['iarea'] = selective_df['iarea'].reset_index(drop=True)

# --- 3. Improved Biplot Visualization ---
def biplot(scores_df, loadings, feature_names, pca_obj, top_n=10):
    """
    Creates a PCA biplot with loading vectors scaled to the plot boundaries, showing all arrows but only labeling the top N contributing variables in red.

    Parameters:
    - scores_df (pd.DataFrame): DataFrame with PCA scores (PC1, PC2).
    - loadings (np.ndarray): The PCA loadings (eigenvectors).
    - feature_names (list): List of feature names corresponding to loadings.
    - pca_obj (PCA): The fitted PCA object from scikit-learn.
    - top_n (int): Number of top contributing variables to show labels for (in red).
    """
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(8, 8))

    # --- Plot the scores (projected data) ---
    sns.scatterplot(
        data=scores_df, x='PC1', y='PC2', ax=ax,
        alpha=0.6, s=60, edgecolor='white', linewidth=0.5, 
        hue='selectivity',palette=selectivity_palette
    )

    # --- Find top N contributing variables ---
    contrib = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
    top_idx = np.argsort(contrib)[-top_n:]

    # --- Plot the loading vectors as arrows ---
    max_score_val = max(np.abs(scores_df['PC1']).max(), np.abs(scores_df['PC2']).max()) * 0.95
    max_loading_val = np.abs(loadings).max()
    scale = max_score_val / max_loading_val * 1.05

    for i, feature in enumerate(feature_names):
        x_vec = loadings[i, 0] * scale
        y_vec = loadings[i, 1] * scale
        ax.arrow(0, 0, x_vec, y_vec,
                 head_width=0.9, head_length=0.9,
                 color='blue', alpha=0.3,linewidth=0.3,
                 length_includes_head=True)
        if i in top_idx:
            ax.text(x_vec * 1.1, y_vec * 1.1, feature,
                    color='red', ha='center', va='center', fontsize=8)
        # Optionally, uncomment below to show all labels in black for non-top contributors
        # else:
        #     ax.text(x_vec * 1.15, y_vec * 1.15, feature,
        #             color='k', ha='center', va='center', fontsize=8)

    explained_var_ratio = pca_obj.explained_variance_ratio_
    ax.set_xlabel(f'PC1 ({explained_var_ratio[0]*100:.1f}% var)', fontsize=14)
    ax.set_ylabel(f'PC2 ({explained_var_ratio[1]*100:.1f}% var)', fontsize=14)
    ax.set_title(f'PCA on Selective Neurons, top {top_n} variable contributions', fontsize=14)
    ax.axhline(0, ls='--', color='grey', linewidth=0.8)
    ax.axvline(0, ls='--', color='grey', linewidth=0.8)
    plt.grid(False)
    plt.tight_layout()
    plt.show()


# --- Call the function to create the plot ---
# biplot(scores_df, loadings, feature_names, pca, top_n=348)

In [ ]:
# # 1. Data Preprocessing
# scaler = StandardScaler()
# data_scaled = scaler.fit_transform(selective_df.drop(columns=['dp','xpos','ypos','iarea','selectivity'], inplace=False))

# # 2. PCA
# n_components = 2
# pca = PCA(n_components=n_components)
# scores = pca.fit_transform(data_scaled)

# # 3. Create a dataframe with the scores
# scores_df = pd.DataFrame(scores, columns=['PC1', 'PC2'])

# # 4. Prepare biplot: get eigenvectors (loadings)
# loadings = pca.components_.T  # shape: (n_features, n_components)
# feature_names = selective_df.drop(columns=['dp','xpos','ypos','iarea','selectivity']).columns

# # 5. Plot the scores and eigenvectors (biplot)
# plt.figure(figsize=(12, 10))
# ax = plt.gca()

# # Scatterplot of projected data
# sns.scatterplot(data=scores_df, x='PC1', y='PC2', ax=ax, alpha=0.7, s=50)

# # Plot eigenvectors (loadings) as arrows
# arrow_length, text_pos = 200.0, 3.8  # Move text further from arrow head
# for i, (x, y) in enumerate(loadings):
#     ax.arrow(0, 0, x*arrow_length, y*arrow_length, color='k', alpha=0.7, head_width=0.07, head_length=0.07, linewidth=1)
#     # Only label a few top features for clarity
#     if i < 10:
#         ax.text(x*text_pos, y*text_pos, feature_names[i], color='k', ha='center', va='center', fontsize=12)

# ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
# ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
# ax.set_title('PCA Biplot of Selective Neurons')
# plt.grid(True)
# plt.tight_layout()
# plt.show()

In [ ]:
pd.DataFrame(dfs[0])

In [ ]:
# import numpy as np
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt
# import pandas as pd
# import seaborn as sns

# # 1. Data Preprocessing
# scaler = StandardScaler()
# selective_df_pca = selective_df.drop(columns=['dp','xpos','ypos','selectivity','iarea'], inplace=False)
# selective_df_pca.columns = selective_df_pca.columns.astype(str)
# data_scaled = scaler.fit_transform(selective_df_pca.T)

# # 2. PCA
# n_components = 2
# pca = PCA(n_components=n_components)
# scores = pca.fit_transform(data_scaled)

# # 3. Create a dataframe with the scores
# scores_df = pd.DataFrame(scores, columns=['PC1', 'PC2'])
# scores_df['trialID'] = pd.DataFrame(data_scaled).index
# scores_df['trialStim'] = dfs[0]['TrialStim'].reset_index(drop=True)
# scores_df['TrialStim'] = dfs[0]['TrialStim'].reset_index(drop=True) #if 'selectivity' not in scores_df else scores_df['selectivity']

# # 4. Plot: 1 row, 2 columns
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Left: seaborn scatterplot colored by selectivity
# sns.scatterplot(
#     x='PC1', y='PC2', s=15, data=scores_df, alpha=0.8, hue='TrialStim', ax=axes[0]
# )
# axes[0].set_xlabel('PC1')
# axes[0].set_ylabel('PC2')
# axes[0].set_title('PCA: Selective Neurons by Trial Stimulus')
# axes[0].legend(title='Stimulus', fontsize=8, title_fontsize=9)
# axes[0].grid(False)

# # Right: scatter with colorbar for trialID
# cmap = plt.cm.get_cmap('cividis')
# sc = axes[1].scatter(
#     scores_df['PC1'], scores_df['PC2'],
#     c=scores_df['trialID'], cmap=cmap, s=15, alpha=0.8
# )
# axes[1].set_xlabel('PC1')
# axes[1].set_ylabel('PC2')
# axes[1].set_title('PCA: Selective Neurons by Trial Number')
# cbar = plt.colorbar(sc, ax=axes[1], shrink=0.8)
# cbar.set_label('Trial number', fontsize=8)
# cbar.ax.tick_params(labelsize=8)
# axes[1].grid(False)

# plt.tight_layout()
# plt.show()

In [ ]:
# --- Call the function to create the plot ---
# biplot(scores_df, loadings, feature_names, pca, top_n=100)

Visualizing the projection of each trial's neuronal activity onto the two principal components (PC1 and PC2).

In [ ]:
selective_df

In [ ]:
# # Filter for selectivity and then sort by 'area_name' so similar strings are grouped together
# selective_df = trial_means_merged[trial_means_merged['selectivity'].isin(['circle1','leaf1'])]
# # Transpose the data
df_t = selective_df[spikes_for_pcs.columns].T

# Extract condition and trial number from the index
df_t = df_t.reset_index()
df_t['condition'] = df_t['index'].apply(lambda x: 'leaf' if 'l_' in x else ('circle' if 'c_' in x else None))
df_t['trial_num'] = df_t['index'].str.extract(r'(?:l_|c_)(\d+)').astype(int)

# # More pythonic and efficient way to exclude columns without dropping them:
# df_t_excluded = df_t.drop(columns=['index', 'condition', 'trial_num'])
# df_t_excluded

# # PCA
# n_components = 2
# pca = PCA(n_components=n_components)
# scores = pca.fit_transform(df_t_excluded)
# results_scores = pd.DataFrame(scores)
# results_scores['condition'] = df_t['condition']
# results_scores['trial_num'] = df_t['trial_num']

# # Plot the projection colored by 'trial_num'
# plt.figure(figsize=(8,6))
# ax = sns.scatterplot(data=results_scores, x=0, y=1, hue='condition', palette='Set2', s=15)

# # Add vector loadings (arrows) for the top contributing features
# # Get loadings (components) and feature names
# loadings = pca.components_.T
# feature_names = df_t_excluded.columns

# # Choose top N features by vector magnitude
# N = 100
# loading_magnitudes = np.linalg.norm(loadings, axis=1)
# top_indices = np.argsort(loading_magnitudes)[-N:]

# # Compute a proper scale for the arrows based on the data range
# # We'll scale the arrows so the longest loading vector is a fraction of the plot's axis range
# x_range = results_scores[0].max() - results_scores[0].min()
# y_range = results_scores[1].max() - results_scores[1].min()
# max_loading = loading_magnitudes[top_indices].max()
# arrow_length = 0.25 * min(x_range, y_range)  # arrows will be at most 25% of axis range
# arrow_scale = arrow_length / max_loading if max_loading > 0 else 1

# for i in top_indices:
#     x_vec, y_vec = loadings[i, 0], loadings[i, 1]
#     ax.arrow(0, 0, x_vec*arrow_scale, y_vec*arrow_scale, 
#              color='red', alpha=0.7, head_width=0.03*arrow_length, head_length=0.05*arrow_length, linewidth=2)
#     ax.text(x_vec*arrow_scale*1.15, y_vec*arrow_scale*1.15, feature_names[i], 
#             color='red', fontsize=6, ha='center', va='center')

# plt.xlabel('PC1')
# plt.ylabel('PC2')
# plt.title('Projection onto First Two PCs with Top Loadings')
# plt.tight_layout()
# plt.show()

# # plt.xlabel('PC1')
# # plt.ylabel('PC2')
# # plt.title('Projection onto First Two PCs colored by area_name')
# # # Create a legend with area_name labels
# # handles, _ = scatter.legend_elements(prop="colors")
# # unique_areas = selective_df['selectivity'].unique()
# # plt.legend(handles, unique_areas, title="area_name", bbox_to_anchor=(1.05, 1), loc='upper left')
# # plt.tight_layout()
# # plt.show()


In [ ]:
df_t

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

# 1. Data Preprocessing
# scaler = StandardScaler()
# data_scaled = scaler.fit_transform(trial_means_merged[spike_cols].T)
selective_df_pc = selective_df.sort_values('area_name')

# 2. Separate Data by Condition
leaf_cols = [col for col in selective_df_pc.columns if col.startswith('l_')]
circle_cols = [col for col in selective_df_pc.columns if col.startswith('c_')]
meta_cols = ['dp', 'xpos', 'ypos', 'iarea', 'area_name', 'selectivity']

data_leaf = selective_df_pc[leaf_cols].T
data_circle = selective_df_pc[circle_cols].T

# 3. Downsample Leaf Condition
num_trials_circle = data_circle.index.nunique()
num_trials_leaf = data_leaf.index.nunique()
# indices_leaf = np.random.choice(data_leaf.shape[0], num_trials_circle, replace=False)
# print(indices_leaf)
# data_leaf_downsampled = data_leaf.iloc[indices_leaf] # Use .iloc for integer indexing

# 4. Combine Downsampled Leaf and Circle Data
# Stack the downsampled data vertically (trials as rows)
# combined_data = np.vstack([data_leaf_downsampled, data_circle])

# 5. PCA
n_components = 2
pca = PCA(n_components=n_components)
# data_scaled = scaler.fit_transform(combined_data)

# 6. Fit and Transform the Combined Data
scores = pca.fit_transform(selective_df_pc[spikes_for_pcs.columns].T)
results_scores = pd.DataFrame(scores, columns=['PC1', 'PC2'])
results_scores['condition'] = df_t['condition']
results_scores['trial_num'] = df_t['trial_num']
# Align PC1 sign
# s = np.sign(results_scores.loc[results_scores.condition=='circle','PC1'].mean()
#             - results_scores.loc[results_scores.condition=='leaf','PC1'].mean()) or 1.0
# results_scores['PC1_aligned'] = s * results_scores['PC1']

# Loadings DF (neuron-level)
loadings = pca.components_.T  # shape: neurons × components
loadings_df = pd.DataFrame(loadings,
                           columns=['PC1_loading','PC2_loading'])
# loadings_df['PC1_loading_aligned'] = s * loadings_df['PC1_loading']

# 7. Separate Scores for Leaf and Circle
scores_leaf_proj = scores[:num_trials_leaf]
scores_circle_proj = scores[num_trials_circle:]

# 8. Create a single figure with 3 subplots using plt.subplots:
#    Left column: biplot (top) and biplot2 (bottom), sharing x axis
#    Right column: PC1 eigenvectors (top), PC2 eigenvectors (bottom), sharing both axes

# import seaborn as sns

explained_variance_ratio = pca.explained_variance_ratio_

# Create subplots: 2 rows, 2 columns
# Left column: biplot (top) and biplot2 (bottom), share x axis
# Right column: PC1 (top) and PC2 (bottom), share both axes
fig, axes = plt.subplots(
    nrows=2, ncols=2, figsize=(18, 6),
    gridspec_kw={'width_ratios': [1, 2], 'height_ratios': [1, 1]}
)

# Assign axes
ax_biplot = axes[0, 0]
ax_biplot2 = axes[1, 0]
ax_pc1 = axes[0, 1]
ax_pc2 = axes[1, 1]

# Add panel labels (A, B, C, D) to each subplot in the upper left corner
panel_labels = ['(A)', '(B)', '(C)', '(D)']
axes_flat = axes.flatten()
for i, ax in enumerate(axes_flat):
    # Adjust x position for left column subplots for better alignment
    if i % 2 == 0:  # left column
        x_pos = -0.18
    else:           # right column
        x_pos = -0.1
    ax.text(
        x_pos, 1.1, panel_labels[i], transform=ax.transAxes,
        fontsize=18, fontweight='bold', va='top', ha='left'
    )

# Share x axis for left column
ax_biplot2.sharex(ax_biplot)
# Share both axes for right column
ax_pc2.sharex(ax_pc1)
ax_pc2.sharey(ax_pc1)

# Hide x-axis labels and ticks for the top row plots (ax_biplot and ax_pc1)
plt.setp(ax_biplot.get_xticklabels(), visible=False)
ax_biplot.xaxis.get_offset_text().set_visible(False)
ax_biplot.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=False)

plt.setp(ax_pc1.get_xticklabels(), visible=False)
ax_pc1.xaxis.get_offset_text().set_visible(False)
ax_pc1.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=False)

# Biplot
# Define a colorblind-friendly palette for selectivity
selectivity_palette = {
    'leaf': '#2F9C92',    # blue
    'circle': '#E4C16E',  # vermillion
}
sns.scatterplot(data=results_scores, x='PC1', y='PC2', hue='condition', palette=selectivity_palette, s=50, ax=ax_biplot)
# Place legend outside the plot, upper right, not overlapping
ax_biplot.legend(title='Condition', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
ax_biplot.set_xlabel(f'PC1 ({explained_variance_ratio[0]*100:.1f}%)')
ax_biplot.set_ylabel(f'PC2 ({explained_variance_ratio[1]*100:.1f}%)')
ax_biplot.set_title('Biplot with Differential Activation by Condition')

sns.scatterplot(data=results_scores, x='PC1', y='PC2', hue='trial_num', palette='viridis', s=50, ax=ax_biplot2)
# Place legend outside the plot, lower right, not overlapping
ax_biplot2.legend(title='Trial #', bbox_to_anchor=(1.02, 0.69), loc='center left', borderaxespad=0.)

ax_biplot2.set_xlabel(f'PC1 ({explained_variance_ratio[0]*100:.1f}%)')
ax_biplot2.set_ylabel(f'PC2 ({explained_variance_ratio[1]*100:.1f}%)')
ax_biplot2.set_title('Biplot with Differential Activation by Trial Number')

# PC1 eigenvector loadings (bar plot)
eigenvectors = pca.components_.T  # Shape: (n_features, n_components)

# Color bars by neuron area using 'area_name' from trial_means_merged
if isinstance(trial_means_merged, pd.DataFrame) and 'area_name' in trial_means_merged.columns:
    try:
        neuron_ids = data_leaf.columns
    except Exception:
        neuron_ids = None
    if neuron_ids is None or len(neuron_ids) != eigenvectors.shape[0]:
        neuron_ids = list(range(eigenvectors.shape[0]))
    area_labels = pd.Series(trial_means_merged['area_name']).reindex(neuron_ids)
    area_labels = area_labels.fillna('Unknown').astype(str)
    unique_areas = pd.unique(area_labels).tolist()
    cmap = plt.get_cmap('Set2')
    area_to_color = {area: cmap(i % 20) for i, area in enumerate(unique_areas)}
    bar_colors = [area_to_color.get(area, 'gray') for area in area_labels]
else:
    bar_colors = 'C0'
    area_to_color = {}
    unique_areas = []

ax_pc1.bar(range(len(eigenvectors[:, 0])), eigenvectors[:, 0], color=bar_colors)

# ax_pc1.set_xlabel('Selective Neuron Index')
ax_pc1.set_ylabel('PC1 Loading')
ax_pc1.set_title('PC1 Neural Contribution by Brain Area')
if len(unique_areas) > 0:
    legend_handles = [Patch(facecolor=area_to_color[a], label=str(a)) for a in unique_areas]
    # Place legend outside the plot, upper right, not overlapping
    ax_pc1.legend(handles=legend_handles, title='Area', bbox_to_anchor=(1.105, 1), loc='upper right')
ax_pc1.set_xlim(-20, 3900)
# PC2 eigenvector loadings (bar plot)
ax_pc2.bar(range(len(eigenvectors[:, 1])), eigenvectors[:, 1], color=bar_colors)

ax_pc2.set_xlabel('Selective Neuron Index')
ax_pc2.set_ylabel('PC2 Loading')
ax_pc2.set_title('PC2 Neural Contribution by Brain Area')
ax_pc2.set_xlim(-20, 3900)

fig.tight_layout()#rect=[0, 0, 0.92, 1])  # leave space for legends on the right
plt.savefig('results/pca_differential_activation_by_area.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

# 9. Calculate Differential Activation (Averaged)
# differential_PC1 = np.mean(scores_leaf_proj[:, 0]) - np.mean(scores_circle_proj[:, 0])
# differential_PC2 = np.mean(scores_leaf_proj[:, 1]) - np.mean(scores_circle_proj[:, 1])

# print(f"Differential Activation (PC1): {differential_PC1:.4f}")
# print(f"Differential Activation (PC2): {differential_PC2:.4f}")

# # 10. Visualize Eigenvectors (now plotted in the right column of the figure above)

# # 11. Thresholding for Selective Neurons (Considerations)
# # Because each dot represents a trial, not a neuron, it doesn't make sense to directly threshold
# # the loadings to identify "selective neurons" here.
# # Instead, you can look at the *average* activity of neurons with high loadings on PC2
# # (the primary separator) for each condition.
# threshold = 0.1  # Example threshold
# selective_neurons_pc2 = np.where(np.abs(eigenvectors[:, 1]) > threshold)[0]
# print(f"Number of neurons with PC2 loading > {threshold}: {len(selective_neurons_pc2)}")

In [ ]:
# get the trial info dataframe
trial_info_df = pd.DataFrame(dfs[0])
trial_info_df

In [ ]:
results_scores#.sort_values('trial_num')

In [ ]:
# Assuming your dataframes are called df1 (with 'trInd') and df2 (with 'trial_num')
merged_scores_df = pd.merge(trial_info_df, results_scores, left_on='trInd', right_on='trial_num', how='inner')
merged_scores_df

In [ ]:
merged_scores_df

In [ ]:
merged_scores_df.columns#[merged_scores_df['condition']=='circle']

In [ ]:
print(merged_scores_df.isna().sum().sort_values(ascending=False))
# some speed related metrics will be missing for circle trials 
# because the rewardPos and rewardFrame times are non-existent
# df[df['TrialStim']=='circle1'].count()

In [ ]:
merged_scores_df[['SoundTime','RewTime','SoundTimeDelay']].describe()

In [ ]:
secs = 86400.0
eps = 1e-6
df = merged_scores_df.copy()

# time (days → seconds)
for c in ['Trial_start_time','Trial_end_time','SoundTime','RewTime','SoundTimeDelay']:
    if c in df: df[c] = pd.to_numeric(df[c], errors='coerce')

df['trial_duration_s'] = (df.Trial_end_time - df.Trial_start_time) * secs
df['time_to_sound_s']  = (df.SoundTime      - df.Trial_start_time) * secs
df['time_to_reward_s'] = (df.RewTime        - df.Trial_start_time) * secs

# positions / progress
df['pos_at_sound'] = df.get('SoundPos')
df['pos_at_reward'] = df.get('RewPos')
df['delta_pos'] = df['pos_at_reward'] - df['pos_at_sound']
if 'delta_pos' in df:
    denom = np.nanmax(np.abs(df['delta_pos'].values)) or 1.0
    df['progress_ratio'] = df['delta_pos'] / denom

# speeds (avoid sound→reward division by ~0)
df['avg_speed_whole']    = df['pos_at_reward'] / np.maximum(df['trial_duration_s'], eps)
df['speed_start_to_reward'] = df['pos_at_reward'] / np.maximum(df['time_to_reward_s'], eps)
# Optional (likely ~0): drop later if near-constant
df['speed_sound_to_reward'] = df['delta_pos'] / np.maximum((df['RewTime'] - df['SoundTime']) * secs, eps)

# trial history
df = df.sort_values('trInd').reset_index(drop=True)

# ensure isRew is 0/1
df['isRew'] = pd.to_numeric(df['isRew'], errors='coerce').fillna(0).astype(int)

# reward_prev: 1 if previous trial was rewarded, else 0
df['reward_prev'] = df['isRew'].shift(1, fill_value=0)

# inter_leaf_idx: running count of consecutive leaf trials (resets to 0 after non-leaf)
# leaf_flag = df['TrialStim'].astype(str).str.lower().str.startswith('leaf')

# inter_leaf_idx = []
# count = -1
# for is_leaf in leaf_flag:
#     if is_leaf:
#         count += 1
#     else:
#         count = -1
#     inter_leaf_idx.append(0 if count == -1 else count)
# df['inter_leaf_idx'] = inter_leaf_idx

# quick cleanup: drop obvious junk
drop_cols = [c for c in ['speed_sound_to_reward'] if c in df and df[c].std(skipna=True) < 1e-9]
df = df.drop(columns=drop_cols, errors='ignore')

In [ ]:
df

In [ ]:
df[['SoundPos','pos_at_reward','pos_at_sound']].describe()

In [ ]:
df_analysis = df.copy()

leaf_ix = df_analysis['TrialStim'].str.lower().str.startswith('leaf1')
rewarded = (df_analysis['isRew']==1) if 'isRew' in df_analysis else df_analysis['RewardFr'].notna()
valid_speed = leaf_ix & rewarded

base_preds  = [c for c in ['trInd','pos_at_reward','reward_prev'] if c in df.columns]
speed_preds = [c for c in ['time_to_reward_s','avg_speed_whole','speed_start_to_reward'] if c in df.columns]
all_preds    = base_preds + speed_preds
all_preds

In [ ]:
import math

ncols = 3
nplots = len(all_preds)
nrows = math.ceil(nplots / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols, 3*nrows))
axes = axes.flatten() if nplots > 1 else [axes]

plot_idx = 0
for col in all_preds:
    sub = df_analysis.loc[leaf_ix, ['PC1', col]].dropna()
    # print(sub)
    if sub.empty: 
        continue
    ax = axes[plot_idx]
    sns.scatterplot(data=sub, x=col, y='PC1', alpha=0.35, s=12, ax=ax)
    sns.regplot(data=sub, x=col, y='PC1', lowess=True, scatter=False, color='crimson', ax=ax)
    ax.set_title(f'Leaf: PC1 vs {col}')
    plot_idx += 1

# Hide any unused axes
for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
import math

def plot_binned_mean_subplot(ax, df_in, x, y='PC1', nbins=12):
    d = df_in[[x, y]].dropna()
    if d.empty: 
        ax.set_visible(False)
        return
    # If all values are the same, binning will fail. Just plot a single point.
    if np.isclose(d[x].max(), d[x].min()):
        ax.scatter([d[x].iloc[0]], [d[y].mean()], s=30)
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.set_title(f'Leaf: binned {y} by {x}')
        return
    bins = np.linspace(d[x].min(), d[x].max(), nbins+1)
    bins = np.unique(bins)
    if len(bins) < 2:
        ax.scatter([d[x].iloc[0]], [d[y].mean()], s=30)
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.set_title(f'Leaf: binned {y} by {x}')
        return
    d = d.assign(bin=pd.cut(d[x], bins=bins, include_lowest=True, duplicates='drop'))
    g = d.groupby('bin', observed=False)[y].agg(['mean','std','count'])
    sem = g['std'] / np.sqrt(g['count'].clip(lower=1))
    xs = [b.mid for b in g.index.categories]
    ax.errorbar(xs, g['mean'], yerr=sem, fmt='-o', markersize=3, capsize=2)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'Leaf: binned {y} by {x}')

# Set up 3 columns and as many rows as needed
ncols = 3
nplots = sum(col in df_analysis.columns for col in all_preds)
nrows = math.ceil(nplots / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4*ncols, 3*nrows))
axes = axes.flatten() if nplots > 1 else [axes]

plot_idx = 0
for col in all_preds:
    if col in df_analysis.columns:
        plot_binned_mean_subplot(axes[plot_idx], df_analysis.loc[leaf_ix], col)
        plot_idx += 1

# Hide any unused axes
for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = [c for c in all_preds if c in df_analysis.columns]
corr = df_analysis.loc[leaf_ix, corr_cols].corr(method='spearman', min_periods=30)
plt.figure(figsize=(1.0*len(corr_cols)+2, 1.0*len(corr_cols)+2))
sns.heatmap(
    corr, 
    vmin=-1, vmax=1, cmap='vlag', square=True, 
    cbar_kws={'shrink':0.8},
    annot=True, 
    fmt=".2f", 
    annot_kws={"size":8, "color":"gray"}
)
plt.title('Leaf predictors: Spearman correlation')
plt.tight_layout(); plt.show()

# Optional: list strongest Leaf correlations with PC1
pc1_corr = df_analysis.loc[leaf_ix, ['PC1'] + corr_cols].corr(method='spearman').loc['PC1', corr_cols].sort_values(key=np.abs, ascending=False)
print(pc1_corr)

In [ ]:
from scipy.stats import kendalltau
for col in all_preds:
    if col in df_analysis:
        a = df_analysis.loc[leaf_ix, ['PC1', col]].dropna()
        if len(a) > 30:
            tau, p = kendalltau(a[col], a['PC1'])
            print(f'Leaf Kendall tau(PC1, {col}) = {tau:.3f} (p={p:.3f})')